# Quickstart

This notebook moves the attenuator, takes data, reads it back, and writes and scores an offset method.

## 1. Set up

The first cell loads everything the notebook uses: the attenuator `att`, MFX's `attenuator_scan` and `get_run`, psana's
`DataSource`, and a function to take pedestals. With `SIMULATE = True` they come from the simulator, and with
`SIMULATE = False` they are the real ones at MFX, so the rest of the notebook works for both. At MFX the attenuator and
DAQ live in hutch-python and psana runs on S3DF, so there the acquisition and the analysis run in separate sessions.

In [ ]:
import numpy as np
from jf_twin import gain_range        # reads the gain stage from the raw data, works on real data too

SIMULATE = True
EXP = "quickstart"                    # at MFX, the experiment name
DETNAME = "jungfrau"                  # at MFX, check the detector name with detnames exp=...,run=...

if SIMULATE:
    from jf_twin import DetectorTwin
    from jf_twin.psana_sim import DataSource
    twin = DetectorTwin(module_shape=(64, 128), exp=EXP, overwrite=True)
    from jf_twin.mfx_sim import attenuator_scan, get_run
    att = twin.att
    take_pedestals = twin.take_pedestals
else:
    from psana import DataSource
    from mfx.db import att
    from mfx.attenuator_scan import attenuator_scan
    from mfx.macros import get_run
    from mfx.bash_utilities import takepeds, makepeds

    def take_pedestals():
        takepeds()                    # dark run
        makepeds(username="...")      # process it and deploy the pedestals

## 2. Move the attenuator

Calling `att` with a transmission moves the blades, and `att.position` reads back the transmission it reached.

In [ ]:
att(1e-2, wait=True)
print(att.position)

In [ ]:
# Set the transmission to 1e-3 and print the transmission the attenuator reaches.

## 3. Take data

`take_pedestals()` takes the dark run. After that, `attenuator_scan` moves the attenuator through a list of
transmissions and records `duration` seconds at each, all in one run. By default it doesn't record and repeats the
scan `5` times, so we set `record=True` and `runs=1`. `get_run()` then gives the number of the run.

In [ ]:
take_pedestals()

attenuator_scan(transmissions=list(np.geomspace(3e-4, 1, 25)), duration=1, record=True, runs=1)
scan = get_run()
print("run", scan)

In [ ]:
# Take another scan with a different range, number of transmissions or duration.

## 4. Read the data

A run has one event per pulse. The transmission isn't recorded, but the events come in the order of the scan, with
`duration` seconds at each transmission.

In [ ]:
myrun = next(DataSource(exp=EXP, run=scan).runs())
det, bmmon = myrun.Detector(DETNAME), myrun.Detector("MfxDg2BmMon")      # beam monitor after the attenuator

evt = next(myrun.events())
raw = det.raw.raw(evt)
print("beam monitor I [J]:   ", bmmon.raw.totalIntensityJoules(evt))
print("raw frame:            ", raw.shape, raw.dtype)
print("gain stage:           ", gain_range(raw)[0, 0, :5])
print("calibrated energy E:  ", det.raw.calib(evt)[0, 0, :5])

In [ ]:
# Plot the calibrated energy of one pixel against the beam monitor reading I over the whole scan, coloured by gain stage.

## 5. Write and score a method

A method reads a run and returns `pixel_offset`, with one value in ADU for every gain stage and pixel. It has the same
shape as the pedestals, and G0 stays at `0`.

In [ ]:
def my_offsets(exp, run):
    myrun = next(DataSource(exp=exp, run=run).runs())
    det = myrun.Detector(DETNAME)
    offsets = np.zeros_like(det.calibconst["pedestals"][0])     # (stage, module, row, column)
    # your analysis goes here
    return offsets

In the simulator, `twin.score` compares offsets with the true ones and returns a score in percent, where `0` is
perfect. At MFX the true offsets are unknown, so there is no score. Min/max from NoGainNoPain is the baseline.

In [ ]:
from protocols import minmax

if SIMULATE:
    for name, offsets in [("no offsets", None),
                          ("min/max", minmax.pixel_offset(EXP, scan, detname=DETNAME)),
                          ("mine", my_offsets(EXP, scan))]:
        print(f"{name:12s} {twin.score(offsets, verbose=False)['score']:.2f} %")

`det.raw.calib` applies only the pedestals and gains, so to see what your offsets do, subtract `O / G` from the
calibrated energy, using the stage each pixel is in.

In [ ]:
# Check your offsets with E / I, as described in the README.

## 6. Compare on a fresh scan

When your method works here, move it into `protocols/my_method.py`. Then `python scripts/compare.py` takes a new scan
and scores every method in `protocols` next to min/max.